# Baseline Model Evaluation: Pre-Fine-Tuning Performance

## Overview
This notebook establishes the **baseline performance** of a small language model on the Opinion Mining task **BEFORE** fine-tuning.

### Goals:
1. Evaluate zero-shot capability
2. Check JSON compliance
3. Measure baseline accuracy

This will be used to compare improvements after LoRA fine-tuning.

## Plan: Baseline Testing with First 20 Samples

**Summary**: The test data contains instruction-prompt pairs with ground-truth aspect-level annotations (term + polarity). Create a notebook implementation that: (1) loads and displays first 20 samples, (2) tests the base model on these 20 reviews, (3) compares predictions against ground-truth aspects, and (4) calculates baseline metrics.

### Steps

1. **Parse first 20 test samples** — In [00_baseline_evaluation.ipynb](E:\FCAI-HU\level4\semester2\nlu\Opinion-Mining-LLM\notebooks\00_baseline_evaluation.ipynb):
   - Load `test.jsonl` and extract first 20 records
   - Display sample structure: `input` (review), `output` (ground-truth aspects with term/polarity)
   - Show data characteristics (review length, aspect count distribution)

2. **Load base model without adapter** — Use `src/model_loader.load_model()` and `load_tokenizer()` with **no adapter path** (baseline only).

3. **Run zero-shot inference on 20 samples** — For each review in first 20:
   - Call `src/inference.generate_structured_output()` to get predictions
   - Log raw model output and extracted JSON
   - Catch any JSON parsing errors

4. **Validate prediction structure** — Use `src/utils.validate_prediction_structure()` to check JSON validity, count valid vs. invalid outputs.

5. **Compare predictions vs. ground-truth** — Map predicted aspects to ground-truth `term`/`polarity` and calculate:
   - JSON validity rate (%)
   - Aspect extraction precision/recall
   - Polarity accuracy for matched aspects

6. **Save baseline results** — Output metrics table and first 20 predictions to `results/baseline_evaluation.csv` for later comparison with fine-tuned model.

### Further Considerations

1. **Format mismatch** — Ground-truth uses `term`/`polarity`, but inference expects `feature`/`sentiment`. Should we adapt the comparison logic to handle both formats?

2. **Error handling** — If base model fails on JSON parsing for some samples, should we log these as invalid or skip them from metrics?


## 1. Setup: Imports and Configuration


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
%cd /content/drive/MyDrive/Opinion-Mining-LLM/src

/content/drive/MyDrive/Opinion-Mining-LLM/src


In [8]:

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import torch

from model_loader import load_model,load_tokenizer, DEFAULT_BASE_MODEL
from inference import generate_structured_output
from utils import validate_prediction_structure

# Configuration
BASE_MODEL_NAME = DEFAULT_BASE_MODEL
NUM_SAMPLES = 20
TEST_DATA_PATH = Path("../data/Labeled-data/test_labeled_v2.json")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"✓ Base Model: {BASE_MODEL_NAME}")
print(f"✓ Number of samples to test: {NUM_SAMPLES}")
print(f"✓ Test data path: {TEST_DATA_PATH}")


✓ Base Model: Qwen/Qwen2.5-1.5B-Instruct
✓ Number of samples to test: 20
✓ Test data path: ../data/Labeled-data/test_labeled_v2.json


## 2. Load and Inspect First 20 Test Samples


In [9]:
import json

# Load first 20 samples from test.jsonl
test_samples = []
with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
    full_content = f.read()
    all_samples = json.loads(full_content)
    test_samples = all_samples[:NUM_SAMPLES]

print(f"Loaded {len(test_samples)} samples from test data\n")

# Display sample data structure
print("=" * 80)
print("SAMPLE DATA STRUCTURE (First Example)")
print("=" * 80)
sample_example = test_samples[0]
print(f"Keys in record: {sample_example.keys()}\n")
print(f"Input (Review): {sample_example['input']}\n")
print(f"Output (Ground-Truth): {sample_example['prediction']}\n")

Loaded 20 samples from test data

SAMPLE DATA STRUCTURE (First Example)
Keys in record: dict_keys(['input', 'prediction'])

Input (Review): Boot time is super fast, around anywhere from 35 seconds to 1 minute.

Output (Ground-Truth): {"domain":"electronics","aspects":[{"term":"Boot time","polarity":"positive"}]}



In [11]:
4# Analyze data characteristics
reviews = [s['input'] for s in test_samples]
ground_truth_outputs = [json.loads(s['prediction']) for s in test_samples]

# Extract statistics
review_lengths = [len(review.split()) for review in reviews]
aspect_counts = [len(gt.get('aspects', [])) for gt in ground_truth_outputs]

print("\n" + "=" * 80)
print("DATA CHARACTERISTICS")
print("=" * 80)
print(f"Number of samples: {len(test_samples)}")
print(f"Average review length: {np.mean(review_lengths):.1f} words")
print(f"Min review length: {min(review_lengths)} words")
print(f"Max review length: {max(review_lengths)} words")
print(f"Average aspects per review: {np.mean(aspect_counts):.2f}")
print(f"Samples with no aspects: {sum(1 for count in aspect_counts if count == 0)}")
print(f"Samples with 1+ aspects: {sum(1 for count in aspect_counts if count > 0)}")

# Show first 5 reviews and their ground-truth aspects
print("\n" + "=" * 80)
print("FIRST 5 REVIEWS AND GROUND-TRUTH ASPECTS")
print("=" * 80)
for idx in range(min(5, len(test_samples))):
    print(f"\n[Sample {idx+1}]")
    print(f"Review: {reviews[idx]}")
    aspects = ground_truth_outputs[idx].get('aspects', [])
    print(f"Ground-truth aspects: {aspects}")



DATA CHARACTERISTICS
Number of samples: 20
Average review length: 12.8 words
Min review length: 4 words
Max review length: 37 words
Average aspects per review: 1.70
Samples with no aspects: 0
Samples with 1+ aspects: 20

FIRST 5 REVIEWS AND GROUND-TRUTH ASPECTS

[Sample 1]
Review: Boot time is super fast, around anywhere from 35 seconds to 1 minute.
Ground-truth aspects: [{'term': 'Boot time', 'polarity': 'positive'}]

[Sample 2]
Review: tech support would not fix the problem unless I bought your plan for 150 plus.
Ground-truth aspects: [{'term': 'tech support', 'polarity': 'negative'}]

[Sample 3]
Review: but in resume this computer rocks!
Ground-truth aspects: [{'term': 'computer', 'polarity': 'positive'}]

[Sample 4]
Review: Set up was easy.
Ground-truth aspects: [{'term': 'Set up', 'polarity': 'positive'}]

[Sample 5]
Review: Did not enjoy the new Windows 8 and touchscreen functions.
Ground-truth aspects: [{'term': 'Windows 8', 'polarity': 'negative'}, {'term': 'touchscreen functi

## 3. Load Base Model (No Fine-tuning Adapter)


In [4]:
pip install -U bitsandbytes>=0.46.1

In [12]:
print("Loading base model and tokenizer...")
print(f"Model: {BASE_MODEL_NAME}")

# Load base model without LoRA adapter
tokenizer = load_tokenizer(base_model_name=BASE_MODEL_NAME)
model = load_model(base_model_name=BASE_MODEL_NAME, adapter_path=None)

print(f"✓ Model loaded successfully")
print(f"✓ Tokenizer loaded successfully")
print(f"✓ Model device: {model.device}")


Loading base model and tokenizer...
Model: Qwen/Qwen2.5-1.5B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Model loaded successfully
✓ Tokenizer loaded successfully
✓ Model device: cuda:0


## 4. Run Zero-Shot Inference on 20 Samples


In [13]:
import json
import time
import torch

# ============================================================================
# Model Evaluation Mode
# ============================================================================

model.eval()

print("Running zero-shot inference on 20 samples...\n")

# ============================================================================
# Tracking Variables
# ============================================================================

predictions = []

valid_json_count = 0
invalid_json_count = 0

generation_times = []

# ============================================================================
# Inference Loop
# ============================================================================

with torch.inference_mode():

    for idx, sample in enumerate(test_samples):

        review = sample["input"]

        # --------------------------------------------------------------------
        # Parse Ground Truth
        # --------------------------------------------------------------------

        try:
            ground_truth = json.loads(sample["prediction"])

        except json.JSONDecodeError:

            ground_truth = None

        # --------------------------------------------------------------------
        # Initialize Variables
        # --------------------------------------------------------------------

        prediction = None
        is_valid = False
        error_message = None

        # --------------------------------------------------------------------
        # Run Inference
        # --------------------------------------------------------------------

        try:

            start_time = time.time()

            # IMPORTANT:
            # generate_structured_output() already:
            # - generates text
            # - extracts JSON
            # - parses JSON
            #
            # So it directly returns a Python dictionary.
            #
            prediction = generate_structured_output(
                model=model,
                tokenizer=tokenizer,
                review_text=review,
                max_new_tokens=256,
            )

            generation_time = time.time() - start_time

            generation_times.append(generation_time)

            # ----------------------------------------------------------------
            # Validate Prediction Structure
            # ----------------------------------------------------------------

            if isinstance(prediction, dict):

                is_valid = validate_prediction_structure(
                    prediction
                )

            else:

                error_message = (
                    f"Invalid prediction type: "
                    f"{type(prediction).__name__}"
                )

            # ----------------------------------------------------------------
            # Update Counters
            # ----------------------------------------------------------------

            if is_valid:
                valid_json_count += 1
            else:
                invalid_json_count += 1

            # ----------------------------------------------------------------
            # Store Prediction
            # ----------------------------------------------------------------

            predictions.append({
                "sample_idx": idx,
                "review": review,
                "prediction": prediction,
                "ground_truth": ground_truth,
                "json_valid": is_valid,
                "generation_time": generation_time,
                "error": error_message,
            })

            # ----------------------------------------------------------------
            # Logging
            # ----------------------------------------------------------------

            print(
                f"[{idx + 1:02d}/{len(test_samples)}] "
                f"✓ JSON Valid: {is_valid}"
            )

            print("\nReview:")
            print(review[:120] + "...")

            print("\nPrediction:")
            print(prediction)

            print("\nGeneration Time:")
            print(f"{generation_time:.2f} sec")

            print("\n" + "-" * 80 + "\n")

        # --------------------------------------------------------------------
        # Exception Handling
        # --------------------------------------------------------------------

        except Exception as e:

            invalid_json_count += 1

            error_message = (
                f"{type(e).__name__}: {str(e)}"
            )

            predictions.append({
                "sample_idx": idx,
                "review": review,
                "prediction": None,
                "ground_truth": ground_truth,
                "json_valid": False,
                "generation_time": None,
                "error": error_message,
            })

            print(
                f"[{idx + 1:02d}/{len(test_samples)}] "
                f"✗ ERROR"
            )

            print("\nReview:")
            print(review[:120] + "...")

            print("\nError:")
            print(error_message)

            print("\n" + "-" * 80 + "\n")



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Running zero-shot inference on 20 samples...

[01/20] ✓ JSON Valid: True

Review:
Boot time is super fast, around anywhere from 35 seconds to 1 minute....

Prediction:
{'review': 'Boot time is super fast, around anywhere from 35 seconds to 1 minute.', 'overall_sentiment': 'positive', 'aspects': [{'feature': 'boot time', 'sentiment': 'positive', 'opinion': 'super fast'}]}

Generation Time:
4.27 sec

--------------------------------------------------------------------------------

[02/20] ✓ JSON Valid: True

Review:
tech support would not fix the problem unless I bought your plan for 150 plus....

Prediction:
{'review': 'tech support would not fix the problem unless I bought your plan for 150 plus.', 'overall_sentiment': 'negative', 'aspects': [{'feature': 'tech support', 'sentiment': 'negative', 'opinion': 'The tech support is unreliable and requires additional payment to resolve issues.'}]}

Generation Time:
3.21 sec

--------------------------------------------------------------------

In [14]:
# ============================================================================
# Final Summary
# ============================================================================

total_samples = len(predictions)

json_validity_rate = (
    valid_json_count / total_samples
    if total_samples > 0
    else 0.0
)

average_generation_time = (
    sum(generation_times) / len(generation_times)
    if generation_times
    else 0.0
)

# ============================================================================
# Final Report
# ============================================================================

print("\n" + "=" * 80)
print("INFERENCE SUMMARY")
print("=" * 80)

print(f"\nTotal Samples:")
print(f"  {total_samples}")

print(f"\nJSON Valid Outputs:")
print(
    f"  {valid_json_count}/{total_samples} "
    f"({json_validity_rate * 100:.1f}%)"
)

print(f"\nInvalid Outputs:")
print(
    f"  {invalid_json_count}/{total_samples} "
    f"({(1 - json_validity_rate) * 100:.1f}%)"
)

print(f"\nAverage Generation Time:")
print(f"  {average_generation_time:.2f} sec/sample")

print("=" * 80)


INFERENCE SUMMARY

Total Samples:
  20

JSON Valid Outputs:
  20/20 (100.0%)

Invalid Outputs:
  0/20 (0.0%)

Average Generation Time:
  3.68 sec/sample


## 5. Evaluate Predictions Against Ground-Truth


In [15]:
from collections import defaultdict
import statistics
import json
import time

# ============================================================================
# Helper Functions
# ============================================================================

def normalize_text(text):
    """Normalize text safely."""

    if not isinstance(text, str):
        return ""

    return text.strip().lower()


def extract_aspect_terms(aspects_list):
    """
    Extract normalized aspect terms.

    Supports:
    - {"term": "..."}
    - {"feature": "..."}
    """

    terms = set()

    if not isinstance(aspects_list, list):
        return terms

    for aspect in aspects_list:

        if not isinstance(aspect, dict):
            continue

        term = (
            aspect.get("feature")
            or aspect.get("term")
        )

        normalized_term = normalize_text(term)

        if normalized_term:
            terms.add(normalized_term)

    return terms


def extract_aspect_sentiments(aspects_list):
    """
    Extract:
        aspect_term -> list[sentiments]

    Supports:
    - sentiment
    - polarity
    """

    sentiment_map = defaultdict(list)

    if not isinstance(aspects_list, list):
        return dict(sentiment_map)

    for aspect in aspects_list:

        if not isinstance(aspect, dict):
            continue

        term = (
            aspect.get("feature")
            or aspect.get("term")
        )

        sentiment = (
            aspect.get("sentiment")
            or aspect.get("polarity")
        )

        normalized_term = normalize_text(term)
        normalized_sentiment = normalize_text(sentiment)

        if normalized_term and normalized_sentiment:
            sentiment_map[normalized_term].append(
                normalized_sentiment
            )

    return dict(sentiment_map)


def compute_precision(tp, fp):
    """Safe precision computation."""

    denominator = tp + fp

    if denominator == 0:
        return 0.0

    return tp / denominator


def compute_recall(tp, fn):
    """Safe recall computation."""

    denominator = tp + fn

    if denominator == 0:
        return 0.0

    return tp / denominator


def compute_f1(precision, recall):
    """Safe F1-score computation."""

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ============================================================================
# Global Metrics
# ============================================================================

true_positives = 0
false_positives = 0
false_negatives = 0

matched_aspects = 0
correct_sentiments = 0

valid_samples = 0
invalid_samples = 0

total_ground_truth_aspects = 0
total_predicted_aspects = 0

precision_scores = []
recall_scores = []
f1_scores = []

sample_results = []

# ============================================================================
# Evaluation
# ============================================================================

print("\n" + "=" * 90)
print("ASPECT-BASED SENTIMENT ANALYSIS EVALUATION")
print("=" * 90 + "\n")

evaluation_start_time = time.time()

for pred_result in predictions:

    sample_idx = pred_result.get("sample_idx", -1)

    # ------------------------------------------------------------------------
    # Validate prediction
    # ------------------------------------------------------------------------

    if (
        not pred_result.get("json_valid")
        or pred_result.get("prediction") is None
    ):

        invalid_samples += 1

        print(
            f"[Sample {sample_idx + 1:02d}] "
            f"INVALID JSON - Skipped"
        )

        continue

    valid_samples += 1

    # ------------------------------------------------------------------------
    # Load prediction and ground-truth
    # ------------------------------------------------------------------------

    ground_truth = pred_result.get("ground_truth", {})
    prediction = pred_result.get("prediction", {})

    gt_aspects = ground_truth.get("aspects", [])
    pred_aspects = prediction.get("aspects", [])

    total_ground_truth_aspects += len(gt_aspects)
    total_predicted_aspects += len(pred_aspects)

    # ------------------------------------------------------------------------
    # Extract normalized aspect terms
    # ------------------------------------------------------------------------

    gt_terms = extract_aspect_terms(gt_aspects)
    pred_terms = extract_aspect_terms(pred_aspects)

    matched_terms = gt_terms.intersection(pred_terms)

    # ------------------------------------------------------------------------
    # Compute sample TP / FP / FN
    # ------------------------------------------------------------------------

    sample_tp = len(matched_terms)
    sample_fp = len(pred_terms - gt_terms)
    sample_fn = len(gt_terms - pred_terms)

    # Update global counters
    true_positives += sample_tp
    false_positives += sample_fp
    false_negatives += sample_fn

    # ------------------------------------------------------------------------
    # Compute sample metrics
    # ------------------------------------------------------------------------

    sample_precision = compute_precision(sample_tp, sample_fp)
    sample_recall = compute_recall(sample_tp, sample_fn)
    sample_f1 = compute_f1(sample_precision, sample_recall)

    precision_scores.append(sample_precision)
    recall_scores.append(sample_recall)
    f1_scores.append(sample_f1)

    # ------------------------------------------------------------------------
    # Sentiment Accuracy
    # ------------------------------------------------------------------------

    gt_sentiment_map = extract_aspect_sentiments(gt_aspects)
    pred_sentiment_map = extract_aspect_sentiments(pred_aspects)

    sample_correct_sentiments = 0

    for term in matched_terms:

        matched_aspects += 1

        gt_sentiments = gt_sentiment_map.get(term, [])
        pred_sentiments = pred_sentiment_map.get(term, [])

        # Match any predicted sentiment with GT
        if any(
            sentiment in gt_sentiments
            for sentiment in pred_sentiments
        ):
            correct_sentiments += 1
            sample_correct_sentiments += 1

    # ------------------------------------------------------------------------
    # Store detailed results
    # ------------------------------------------------------------------------

    sample_results.append({
        "sample_idx": sample_idx,
        "ground_truth_terms": sorted(gt_terms),
        "predicted_terms": sorted(pred_terms),
        "matched_terms": sorted(matched_terms),
        "precision": sample_precision,
        "recall": sample_recall,
        "f1_score": sample_f1,
        "correct_sentiments": sample_correct_sentiments,
        "matched_aspects": len(matched_terms),
    })

    # ------------------------------------------------------------------------
    # Logging
    # ------------------------------------------------------------------------

    review_preview = (
        pred_result["review"]
        .replace("\n", " ")
        .strip()[:100]
    )

    print(f"[Sample {sample_idx + 1:02d}]")

    print(f"Review:")
    print(f"  {review_preview}...")

    print(f"\nGround Truth:")
    print(f"  {sorted(gt_terms)}")

    print(f"\nPrediction:")
    print(f"  {sorted(pred_terms)}")

    print(f"\nMatched:")
    print(f"  {sorted(matched_terms)}")

    print(f"\nMetrics:")
    print(f"  Precision : {sample_precision:.4f}")
    print(f"  Recall    : {sample_recall:.4f}")
    print(f"  F1-Score  : {sample_f1:.4f}")

    print("-" * 90)

# ============================================================================
# Final Metrics
# ============================================================================

evaluation_time = time.time() - evaluation_start_time

# ---------------------------
# Macro Metrics
# ---------------------------

macro_precision = (
    statistics.mean(precision_scores)
    if precision_scores else 0.0
)

macro_recall = (
    statistics.mean(recall_scores)
    if recall_scores else 0.0
)

macro_f1 = (
    statistics.mean(f1_scores)
    if f1_scores else 0.0
)

# ---------------------------
# Micro Metrics
# ---------------------------

micro_precision = compute_precision(
    true_positives,
    false_positives
)

micro_recall = compute_recall(
    true_positives,
    false_negatives
)

micro_f1 = compute_f1(
    micro_precision,
    micro_recall
)

# ---------------------------
# Sentiment Accuracy
# ---------------------------

sentiment_accuracy = (
    correct_sentiments / matched_aspects
    if matched_aspects > 0
    else 0.0
)

# ============================================================================
# Final Evaluation Report
# ============================================================================

print("\n" + "=" * 90)
print("FINAL EVALUATION REPORT")
print("=" * 90)

print("\nDataset Statistics")
print("-" * 90)
print(f"Valid Samples              : {valid_samples}")
print(f"Invalid Samples            : {invalid_samples}")
print(f"Ground-Truth Aspects       : {total_ground_truth_aspects}")
print(f"Predicted Aspects          : {total_predicted_aspects}")

print("\nMacro Average Metrics")
print("-" * 90)
print(f"Macro Precision            : {macro_precision:.4f}")
print(f"Macro Recall               : {macro_recall:.4f}")
print(f"Macro F1-Score             : {macro_f1:.4f}")

print("\nMicro Average Metrics")
print("-" * 90)
print(f"Micro Precision            : {micro_precision:.4f}")
print(f"Micro Recall               : {micro_recall:.4f}")
print(f"Micro F1-Score             : {micro_f1:.4f}")

print("\nSentiment Classification")
print("-" * 90)
print(f"Sentiment Accuracy         : {sentiment_accuracy:.4f}")

print("\nPerformance")
print("-" * 90)
print(f"Evaluation Time (seconds)  : {evaluation_time:.2f}")

print("=" * 90)


ASPECT-BASED SENTIMENT ANALYSIS EVALUATION

[Sample 01]
Review:
  Boot time is super fast, around anywhere from 35 seconds to 1 minute....

Ground Truth:
  ['boot time']

Prediction:
  ['boot time']

Matched:
  ['boot time']

Metrics:
  Precision : 1.0000
  Recall    : 1.0000
  F1-Score  : 1.0000
------------------------------------------------------------------------------------------
[Sample 02]
Review:
  tech support would not fix the problem unless I bought your plan for 150 plus....

Ground Truth:
  ['tech support']

Prediction:
  ['tech support']

Matched:
  ['tech support']

Metrics:
  Precision : 1.0000
  Recall    : 1.0000
  F1-Score  : 1.0000
------------------------------------------------------------------------------------------
[Sample 03]
Review:
  but in resume this computer rocks!...

Ground Truth:
  ['computer']

Prediction:
  ['computer']

Matched:
  ['computer']

Metrics:
  Precision : 1.0000
  Recall    : 1.0000
  F1-Score  : 1.0000
-------------------------------

## 6. Save Baseline Results


In [17]:
# Create results dataframe
results_data = []

for pred_result in predictions:
    results_data.append({
        'sample_idx': pred_result['sample_idx'],
        'review': pred_result['review'],
        'json_valid': pred_result['json_valid'],
        'error': pred_result['error'],
        'ground_truth_aspects': json.dumps(pred_result['ground_truth'].get('aspects', [])),
        'predicted_aspects': json.dumps(pred_result['prediction'].get('aspects', []) if pred_result['prediction'] else []),
    })

results_df = pd.DataFrame(results_data)
results_csv_path = RESULTS_DIR / "baseline_evaluation.csv"
results_df.to_csv(results_csv_path, index=False)

print(f"✓ Results saved to: {results_csv_path}\n")
print(results_df[['sample_idx', 'json_valid', 'error']].to_string())


✓ Results saved to: ../results/baseline_evaluation.csv

    sample_idx  json_valid error
0            0        True  None
1            1        True  None
2            2        True  None
3            3        True  None
4            4        True  None
5            5        True  None
6            6        True  None
7            7        True  None
8            8        True  None
9            9        True  None
10          10        True  None
11          11        True  None
12          12        True  None
13          13        True  None
14          14        True  None
15          15        True  None
16          16        True  None
17          17        True  None
18          18        True  None
19          19        True  None


## 7. Visualization
